# Step 3 - User Interaction #

In [1]:
import numpy as np
import open3d as o3d
from scipy.spatial import KDTree
from matplotlib.path import Path as MplPath
from typing import List, Tuple, Optional, Dict, Any

from pipeline.vignette_data import ProcessedVignette

In [2]:
# Helper Methods
def _get_projection_params(vignette: ProcessedVignette) -> Dict[str, Any]:
    """
    Extracts and calculates all necessary projection data from a vignette.
    [CORRECTED] This version now correctly scales the camera intrinsics
    to match the resolution of the depth map.
    """
    try:
        meta = vignette.metadata["capture_metadata"]
        
        # --- Get Resolutions ---
        depth_w, depth_h = meta['resolution'][0], meta['resolution'][1]
        
        # We need the original image resolution to calculate the scaling factor.
        # Let's assume it's stored in a key like 'original_resolution'.
        # If not, we will print a warning and assume no scaling is needed.
        if 'original_resolution' in meta:
            original_w, original_h = meta['original_resolution'][0], meta['original_resolution'][1]
        else:
            print("[WARNING] 'original_resolution' not found in metadata. Assuming depth resolution is original. Projection might be inaccurate if this is not the case.")
            original_w, original_h = depth_w, depth_h

        # --- Get Raw Intrinsics and Offset ---
        raw_intrinsics = np.array(meta['camera_intrinsics']['columns']).T
        center_offset = np.array(meta['center_offset'])

        # --- THE FIX: Scale the intrinsics matrix ---
        x_scale = depth_w / original_w
        y_scale = depth_h / original_h

        fx, fy = raw_intrinsics[0, 0], raw_intrinsics[1, 1]
        cx, cy = raw_intrinsics[0, 2], raw_intrinsics[1, 2]
        
        scaled_fx, scaled_fy = fx * x_scale, fy * y_scale
        scaled_cx, scaled_cy = cx * x_scale, cy * y_scale
        
        scaled_K = np.array([
            [scaled_fx, 0, scaled_cx],
            [0, scaled_fy, scaled_cy],
            [0, 0, 1]
        ])
        
        print(f"[DEBUG] Original K matrix principal point: ({cx:.2f}, {cy:.2f})")
        print(f"[DEBUG] Scaled K matrix principal point:   ({scaled_cx:.2f}, {scaled_cy:.2f})")

        return {
            "K": scaled_K, # Use the SCALED matrix
            "offset": center_offset,
            "width": depth_w,
            "height": depth_h
        }
    except KeyError as e:
        raise ValueError(f"Vignette is missing required metadata for projection: {e}")

# --- Helper Function 2: Projecting 3D to 2D ---

def _project_3d_to_uv(points: np.ndarray, params: Dict[str, Any]) -> np.ndarray:
    """Projects 3D points to normalized 2D UV coordinates."""
    # Undo the centering to get points in original camera space
    points_in_camera_space = points + params["offset"]

    # Project points to pixel coordinates
    projected = points_in_camera_space @ params["K"].T
    px = projected[:, 0] / projected[:, 2]
    py = projected[:, 1] / projected[:, 2]

    # Normalize to get UV coordinates
    uv_coords = np.vstack([px / params["width"], py / params["height"]]).T
    return uv_coords

def _trim_and_densify_segment(
    p_start: np.ndarray, 
    p_end: np.ndarray, 
    vignette_kdtree: o3d.geometry.KDTreeFlann,
    all_points: np.ndarray,
    radius: float
) -> Optional[np.ndarray]:
    """
    Finds all points in a point cloud that lie along the 3D line segment 
    between p_start and p_end, within a given radius.
    Returns the points sorted along the segment.
    """
    segment_vector = p_end - p_start
    segment_length = np.linalg.norm(segment_vector)
    if segment_length < 1e-6:
        return None # Zero-length segment

    segment_unit_vector = segment_vector / segment_length
    
    # Step along the line segment and do a radius search at each step
    # The step size is the radius itself to ensure continuous coverage
    num_steps = int(np.ceil(segment_length / radius))
    
    found_indices = set()
    for i in range(num_steps + 1):
        sample_point = p_start + (segment_unit_vector * (i * radius))
        # Find all points in the vignette within the radius of our sample point
        [k, idx, _] = vignette_kdtree.search_radius_vector_3d(sample_point, radius)
        if k > 0:
            found_indices.update(idx)

    if not found_indices:
        return None

    # We have an unordered set of points near the line. Now we sort them.
    found_points = all_points[list(found_indices)]
    
    # Project each found point onto the line segment to get its distance along the line
    vectors_from_start = found_points - p_start
    distances = np.dot(vectors_from_start, segment_unit_vector)
    
    # Sort the points by their projected distance
    sorted_points = found_points[np.argsort(distances)]
    
    return sorted_points


### Interaction 1 - project curves to get 3D curves ###

In [3]:
import numpy as np
import open3d as o3d
from scipy.spatial import KDTree
from typing import List, Tuple, Optional

# (Assuming your ProcessedVignette class and helper functions
# _get_projection_params, _project_3d_to_uv, _resample_uv_curve, and
# _trim_and_densify_segment are defined as before)

def project_curves_to_3d_enhanced(
    vignette: ProcessedVignette, 
    uv_curves: List[List[Tuple[float, float]]],
    follow_surface: bool = False,
    trim_radius: Optional[float] = None,
    validation_radius: float = 0.01 # <-- NEW PARAMETER FOR SURFACE FOLLOWING
) -> List[List[np.ndarray]]:
    """
    Projects 2D UV curves to 3D, handling sparse point clouds correctly.
    [FULLY IMPLEMENTED VERSION]

    Args:
        vignette: The ProcessedVignette object.
        uv_curves: A list of curves to project.
        follow_surface: Toggles the projection mode.
        trim_radius: For Nearest Neighbor mode, densifies the line and keeps it on the surface.
        validation_radius: For Surface Following mode, the 3D distance to check for a nearby point.

    Returns:
        A list of results, one for each input curve. Each result is a LIST of segments (np.ndarray).
        This is because a single input curve might be broken into multiple 3D segments.
    """
    mode_str = "Surface Following" if follow_surface else f"Nearest Neighbor (Trim Radius: {trim_radius})"
    print(f"\n--- Projecting {len(uv_curves)} curves (Mode: {mode_str}) ---")

    try:
        params = _get_projection_params(vignette)
    except (ValueError, KeyError) as e:
        print(f"[ERROR] Could not prepare for projection. {e}")
        return [[] for _ in uv_curves]

    all_results = []

    # --- Build 3D KD-Tree once if needed by either mode ---
    vignette_kdtree_3d = None
    if (not follow_surface and trim_radius is not None) or follow_surface:
        print("[DEBUG] Building 3D KD-Tree of vignette points for validation/trimming...")
        pcd_3d = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(vignette.points))
        vignette_kdtree_3d = o3d.geometry.KDTreeFlann(pcd_3d)

    # --- Process each curve individually ---
    for i, curve in enumerate(uv_curves):
        print(f"  -> Processing input curve #{i+1}...")
        
        if not follow_surface:
            # --- Mode 1: Nearest Neighbor ---
            vignette_uvs = _project_3d_to_uv(vignette.points, params)
            uv_kdtree = KDTree(vignette_uvs)
            query_points = np.array(curve)
            if len(query_points) == 0:
                all_results.append([])
                continue
            
            _, nearest_indices = uv_kdtree.query(query_points)
            anchor_points = vignette.points[nearest_indices]

            if trim_radius is None or vignette_kdtree_3d is None:
                all_results.append([anchor_points]) # Return as a single segment
            else:
                densified_curve = []
                for i in range(len(anchor_points) - 1):
                    segment = _trim_and_densify_segment(
                        anchor_points[i], anchor_points[i+1], vignette_kdtree_3d, vignette.points, trim_radius
                    )
                    if segment is not None:
                        densified_curve.extend(segment)
                all_results.append([np.array(densified_curve)] if densified_curve else [])
        
        else:
            # --- Mode 2: Surface Following (Fully Implemented) ---
            try:
                vignette_folder = vignette.file_path.parent.parent
                depth_path = vignette_folder / "depth.bin"
                depth_data = np.fromfile(depth_path, dtype=np.float32)
                h, w = params['height'], params['width']
                depth_map = depth_data.reshape((h, w))
            except (FileNotFoundError, AttributeError):
                print(f"[ERROR] Could not load depth map. Skipping curve.")
                all_results.append([])
                continue
            
            K_inv = np.linalg.inv(params['K'])
            
            if len(curve) < 2:
                all_results.append([])
                continue

            resampled_uv = _resample_uv_curve(curve, (params['width'], params['height']))
            
            all_segments = []
            current_segment = []
            
            for u, v in resampled_uv:
                px, py = int(u * params['width']), int(v * params['height'])
                if 0 <= py < params['height'] and 0 <= px < params['width']:
                    depth = depth_map[py, px]
                    if depth > 0:
                        point_cam_space = (K_inv @ np.array([px, py, 1])) * depth
                        candidate_point = point_cam_space - params['offset']
                        
                        # VALIDATION STEP: Check if a real point is nearby
                        [k, _, _] = vignette_kdtree_3d.search_radius_vector_3d(candidate_point, validation_radius)
                        
                        if k > 0:
                            # This point is valid, add it to the current segment
                            current_segment.append(candidate_point)
                        else:
                            # This point is in an empty area, which means a break in the curve
                            if len(current_segment) > 1:
                                all_segments.append(np.array(current_segment))
                            current_segment = [] # Reset for the next segment
                    else:
                        # Zero depth also means a break in the curve
                        if len(current_segment) > 1:
                            all_segments.append(np.array(current_segment))
                        current_segment = []
            
            # After the loop, add any remaining segment
            if len(current_segment) > 1:
                all_segments.append(np.array(current_segment))
            
            print(f"    [SUCCESS] Generated {len(all_segments)} 3D segment(s) for this curve.")
            all_results.append(all_segments)

    return all_results

### Updated Visualization Example

def create_line_set_from_segments(list_of_curve_segments: List[List[np.ndarray]], color: List[float]):
    """
    Creates a list of Open3D LineSet objects from a nested list of curve segments.
    """
    line_sets = []
    for curve_result in list_of_curve_segments: # Outer loop for each original curve
        for segment in curve_result:             # Inner loop for each segment of that curve
            if segment is not None and len(segment) > 1:
                pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(segment))
                lines = [[i, i + 1] for i in range(len(segment) - 1)]
                ls = o3d.geometry.LineSet(points=pcd.points, lines=o3d.utility.Vector2iVector(lines))
                ls.paint_uniform_color(color)
                line_sets.append(ls)
    return line_sets

if __name__ == '__main__':
    # 1. Load your vignette
    vignette_file_path = Path("path/to/your/results/raw_vignette.npz")
    vignette_object = ProcessedVignette.load(vignette_file_path)

    # 2. Define a user curve that might cross an empty space
    long_line = [(0.2, 0.5), (0.8, 0.5)]
    user_curves = [long_line]
    
    # 3. Run the fully corrected surface following mode
    # Adjust validation_radius based on your point cloud's density
    results_surface = project_curves_to_3d_enhanced(
        vignette_object, user_curves, follow_surface=True, validation_radius=0.01
    )

    # 4. Visualize
    vignette_pcd = vignette_object.to_open3d()
    
    # Use the updated visualization helper
    lines_surface = create_line_set_from_segments(results_surface, color=[0, 1, 1]) # Cyan
    
    print("\nDisplaying visualization...")
    print("CYAN lines = Surface Following (Validated and Segmented)")
    
    o3d.visualization.draw_geometries([vignette_pcd, *lines_surface])

NameError: name 'Path' is not defined

In [4]:
import numpy as np
import open3d as o3d
from scipy.spatial import KDTree
from typing import List, Tuple, Optional

# (Assuming all your other helper functions and the ProcessedVignette class are defined)



def project_curves_to_3d_enhanced(
    vignette: ProcessedVignette, 
    uv_curves: List[List[Tuple[float, float]]],
    follow_surface: bool = False,
    trim_radius: Optional[float] = None # <-- NEW PARAMETER
) -> List[Optional[np.ndarray]]:
    """
    Projects 2D UV curves to 3D. Includes a new 'trim_radius' parameter
    for the Nearest Neighbor mode to ensure lines stay on the surface.
    """
    mode_str = "Surface Following" if follow_surface else f"Nearest Neighbor (Trim Radius: {trim_radius})"
    print(f"\n--- Projecting {len(uv_curves)} curves (Mode: {mode_str}) ---")

    # ... (Initial setup and error checking is the same) ...
    try:
        params = _get_projection_params(vignette)
        vignette_uvs = _project_3d_to_uv(vignette.points, params)
    except (ValueError, KeyError) as e:
        print(f"[ERROR] Could not prepare for projection. {e}")
        return [None] * len(uv_curves)

    results_3d = []

    if not follow_surface:
        # --- Mode 1: Nearest Neighbor (Now with optional trimming) ---
        uv_kdtree = KDTree(vignette_uvs)
        
        # Build a 3D KD-Tree of the vignette points if we are trimming
        vignette_kdtree_3d = None
        if trim_radius is not None:
            print("[DEBUG] Building 3D KD-Tree for trimming operation...")
            pcd_3d = o3d.geometry.PointCloud()
            pcd_3d.points = o3d.utility.Vector3dVector(vignette.points)
            vignette_kdtree_3d = o3d.geometry.KDTreeFlann(pcd_3d)

        for curve in uv_curves:
            query_points = np.array(curve)
            if len(query_points) == 0:
                results_3d.append(None)
                continue
            
            # 1. Find the sparse anchor points as before
            _, nearest_indices = uv_kdtree.query(query_points)
            anchor_points = vignette.points[nearest_indices]

            if trim_radius is None or vignette_kdtree_3d is None:
                # If not trimming, return the simple straight-line connection
                results_3d.append(anchor_points)
            else:
                # 2. If trimming, process each segment between anchors
                densified_curve = []
                for i in range(len(anchor_points) - 1):
                    p_start, p_end = anchor_points[i], anchor_points[i+1]
                    
                    # Call the new helper to find on-surface points for this segment
                    trimmed_segment = _trim_and_densify_segment(
                        p_start, p_end, vignette_kdtree_3d, vignette.points, trim_radius
                    )
                    
                    if trimmed_segment is not None:
                        densified_curve.extend(trimmed_segment)
                
                if densified_curve:
                    results_3d.append(np.array(densified_curve))
                else:
                    results_3d.append(None) # No points found for any segment
        
    else: # Surface following mode is unchanged
        # ... (your existing `follow_surface=True` logic goes here) ...
        # (For brevity, it is omitted here, but it should be included in your final code)
        print("[DEBUG] Running surface following mode (logic is unchanged)...")
        # (You would have your depth map loading and unprojection loop here)
        # This part is already working correctly so no need to modify it.
        pass # Placeholder for your existing logic
        # For the script to run, I'll paste the logic back in here:
        try:
            vignette_folder = vignette.file_path.parent.parent
            depth_path = vignette_folder / "depth.bin"
            depth_data = np.fromfile(depth_path, dtype=np.float32)
            h, w = params['height'], params['width']
            depth_map = depth_data.reshape((h, w))
        except (FileNotFoundError, AttributeError): return [None] * len(uv_curves)
        K_inv = np.linalg.inv(params['K'])
        for curve in uv_curves:
            if len(curve) < 2: results_3d.append(None); continue
            resampled_uv = _resample_uv_curve(curve, (params['width'], params['height']))
            curve_3d_points = []
            for u, v in resampled_uv:
                px, py = int(u * params['width']), int(v * params['height'])
                if 0 <= py < params['height'] and 0 <= px < params['width']:
                    depth = depth_map[py, px]
                    if depth > 0:
                        point_cam_space = (K_inv @ np.array([px, py, 1])) * depth
                        final_point_3d = point_cam_space - params['offset']
                        curve_3d_points.append(final_point_3d)
            if curve_3d_points: results_3d.append(np.array(curve_3d_points))
            else: results_3d.append(None)


    return results_3d

In [5]:
import numpy as np
import open3d as o3d
from scipy.spatial import KDTree
from typing import List, Tuple, Optional

# (Assuming your ProcessedVignette class and helper functions
# _get_projection_params and _project_3d_to_uv are defined as before)

def _resample_uv_curve(uv_curve: List[Tuple[float, float]], pixel_dims: Tuple[int, int]) -> np.ndarray:
    """
    Resamples a sparse UV curve into a dense one with approximately 1-pixel spacing.
    """
    w, h = pixel_dims
    resampled_points = []
    pixel_curve = np.array(uv_curve) * [w, h]
    for i in range(len(pixel_curve) - 1):
        p1, p2 = pixel_curve[i], pixel_curve[i+1]
        segment_vector = p2 - p1
        segment_length = np.linalg.norm(segment_vector)
        num_steps = max(1, int(np.ceil(segment_length)))
        for step in range(num_steps):
            interp_point = p1 + (segment_vector * (step / num_steps))
            resampled_points.append(interp_point)
    resampled_points.append(pixel_curve[-1])
    return np.array(resampled_points) / [w, h]

def project_curves_to_3d(
    vignette: ProcessedVignette, 
    uv_curves: List[List[Tuple[float, float]]],
    follow_surface: bool = False
) -> List[Optional[np.ndarray]]:
    """
    Projects a list of 2D UV curves onto the 3D point cloud surface.
    [CORRECTED LOGIC & ADDED DEBUGGING]
    """
    print(f"\n--- Projecting {len(uv_curves)} curves (Mode: {'Surface Following' if follow_surface else 'Nearest Neighbor'}) ---")
    try:
        params = _get_projection_params(vignette)
        vignette_uvs = _project_3d_to_uv(vignette.points, params)
        print(f"[DEBUG] Projection params loaded. Camera Intrinsics K:\n{params['K']}")
        print(f"[DEBUG] Center Offset vector: {params['offset']}")
    except (ValueError, KeyError) as e:
        print(f"[ERROR] Could not prepare for projection. {e}")
        return [None] * len(uv_curves)

    results_3d = []

    if not follow_surface:
        # --- Mode 1: Nearest Neighbor ---
        print("[DEBUG] Building KD-Tree for nearest neighbor search...")
        kdtree = KDTree(vignette_uvs)
        for i, curve in enumerate(uv_curves):
            print(f"  -> Processing curve #{i+1}...")
            query_points = np.array(curve)
            if len(query_points) == 0:
                print("    [WARN] Input curve is empty. Skipping.")
                results_3d.append(None)
                continue
            
            _, nearest_indices = kdtree.query(query_points)
            result_curve = vignette.points[nearest_indices]
            results_3d.append(result_curve)
            print(f"    [SUCCESS] Found {len(result_curve)} corresponding 3D points.")
            if len(result_curve) > 0:
                print(f"      - First projected point (centered): {result_curve[0]}")
        
    else:
        # --- Mode 2: Surface Following (Draping) ---
        try:
            vignette_folder = vignette.file_path.parent.parent
            depth_path = vignette_folder / "depth.bin"
            print(f"[DEBUG] Loading depth map from: {depth_path}")
            depth_data = np.fromfile(depth_path, dtype=np.float32)
            h, w = params['height'], params['width']
            depth_map = depth_data.reshape((h, w))
            print(f"[DEBUG] Depth map loaded with shape: {depth_map.shape}")
        except (FileNotFoundError, AttributeError) as e:
            print(f"[ERROR] Could not load depth map for surface following mode: {e}")
            return [None] * len(uv_curves)
            
        K_inv = np.linalg.inv(params['K'])

        for i, curve in enumerate(uv_curves):
            print(f"  -> Processing curve #{i+1}...")
            if len(curve) < 2: 
                print("    [WARN] Input curve has fewer than 2 points. Skipping.")
                results_3d.append(None)
                continue

            resampled_uv = _resample_uv_curve(curve, (params['width'], params['height']))
            print(f"    [DEBUG] Resampled 2D curve to {len(resampled_uv)} points.")
            
            curve_3d_points = []
            for u, v in resampled_uv:
                px, py = int(u * params['width']), int(v * params['height'])
                if 0 <= py < params['height'] and 0 <= px < params['width']:
                    depth = depth_map[py, px]
                    if depth > 0:
                        point_2d_homog = np.array([px, py, 1])
                        point_cam_space = (K_inv @ point_2d_homog) * depth
                        
                        # --- THE FIX ---
                        # To align with the centered vignette points, we must
                        # SUBTRACT the same offset.
                        final_point_3d = point_cam_space - params['offset']
                        curve_3d_points.append(final_point_3d)

            if curve_3d_points:
                result_curve = np.array(curve_3d_points)
                results_3d.append(result_curve)
                print(f"    [SUCCESS] Generated {len(result_curve)} points for the 3D curve.")
                print(f"      - First unprojected point (camera space): {point_cam_space}")
                print(f"      - First final point (centered space): {result_curve[0]}")
            else:
                print("    [WARN] No valid 3D points generated for this curve (likely off-model or in a no-depth area).")
                results_3d.append(None)
                
    return results_3d

In [6]:
from pathlib import Path
import sys
import open3d as o3d

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

VIGNETTE_NAME = "cloth_02"
VIGNETTE_PATH = project_root / "test_data" / VIGNETTE_NAME

rgb_path = VIGNETTE_PATH / "rgb.png"
results_path = VIGNETTE_PATH / "results"

In [7]:
# Load preprocessed vignette for tests
processed_vignette_path = results_path / "processed_vignette.npz"
from pipeline.vignette_data import ProcessedVignette
processed_vignette = ProcessedVignette.load(processed_vignette_path)

Set/updated per-point attribute: 'confidence'.
Set/updated per-point attribute: 'component_id'.
Set/updated per-point attribute: 'curvature'.
Set/updated per-point attribute: 'edgeness'.
Set/updated per-point attribute: 'anisotropy'.
Set/updated per-point attribute: 'planarity'.
Set/updated per-point attribute: 'sphericity'.
Set/updated per-point attribute: 'flow_vectors'.
Set/updated per-point attribute: '2d_edges'.
Set/updated per-point attribute: '2d_texture_detail'.
Set/updated per-point attribute: 'stylized_colors'.
Set/updated per-point attribute: 'plane_id'.
Loaded processed vignette from: /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/cloth_02/results/processed_vignette.npz


In [8]:
# Apply a test curve
line_curve = [(0.1, 0.5), (0.9, 0.5)]
line_curve2 = [(0.1, 0.3), (0.9, 0.3)]
line_curve3 = [(0.1, 0.7), (0.9, 0.7)]
# A V-shape to show connected segments
# v_curve = [(0.45, 0.6), (0.5, 0.7), (0.55, 0.6)]
user_curves = [line_curve, line_curve2, line_curve3]

results_nearest = project_curves_to_3d(processed_vignette, user_curves, follow_surface=False)
results_surface = project_curves_to_3d(processed_vignette, user_curves, follow_surface=True)



--- Projecting 3 curves (Mode: Nearest Neighbor) ---
[DEBUG] Original K matrix principal point: (960.19, 726.85)
[DEBUG] Scaled K matrix principal point:   (128.03, 96.91)
[DEBUG] Projection params loaded. Camera Intrinsics K:
[[182.60874023   0.         128.02589518]
 [  0.         182.60874023  96.91315104]
 [  0.           0.           1.        ]]
[DEBUG] Center Offset vector: [ 0.02277316 -0.00064674  0.46487512]
[DEBUG] Building KD-Tree for nearest neighbor search...
  -> Processing curve #1...
    [SUCCESS] Found 2 corresponding 3D points.
      - First projected point (centered): [-0.23620157 -0.00126348 -0.08287512]
  -> Processing curve #2...
    [SUCCESS] Found 2 corresponding 3D points.
      - First projected point (centered): [-0.23396671 -0.07990346 -0.08687513]
  -> Processing curve #3...
    [SUCCESS] Found 2 corresponding 3D points.
      - First projected point (centered): [-0.23843642  0.07904126 -0.07887511]

--- Projecting 3 curves (Mode: Surface Following) ---
[

In [13]:
geometries_to_draw = []
vignette_pcd = processed_vignette.to_open3d()
geometries_to_draw.append(vignette_pcd)

def create_line_set(curves_3d, color):
    line_sets = []
    for curve in curves_3d:
        if curve is not None and len(curve) > 1:
            pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(curve))
            lines = [[i, i + 1] for i in range(len(curve) - 1)]
            ls = o3d.geometry.LineSet(points=pcd.points, lines=o3d.utility.Vector2iVector(lines))
            ls.paint_uniform_color(color)
            line_sets.append(ls)
    return line_sets

geometries_to_draw.extend(create_line_set(results_nearest, color=[0, 1, 0]))
geometries_to_draw.extend(create_line_set(results_surface, color=[1, 0, 0]))

o3d.visualization.draw_geometries(geometries_to_draw)

Generating Open3D point cloud with 'rgb' colors...


In [ ]:
import open3d as o3d
import numpy as np

geometries_to_draw = []
vignette_pcd = processed_vignette.to_open3d()
geometries_to_draw.append(vignette_pcd)

def create_line_set(curves_3d, color):
    line_sets = []
    for curve in curves_3d:
        if curve is not None and len(curve) > 1:
            pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(curve))
            lines = [[i, i + 1] for i in range(len(curve) - 1)]
            ls = o3d.geometry.LineSet(points=pcd.points, lines=o3d.utility.Vector2iVector(lines))
            ls.paint_uniform_color(color)
            line_sets.append(ls)
    return line_sets

geometries_to_draw.extend(create_line_set(results_nearest, color=[0, 1, 0]))
geometries_to_draw.extend(create_line_set(results_surface, color=[1, 0, 0]))

# --- Use Visualizer instead of draw_geometries ---
vis = o3d.visualization.Visualizer()
vis.create_window(window_name="Thicker Lines", width=1280, height=720)

for g in geometries_to_draw:
    vis.add_geometry(g)

render_opt = vis.get_render_option()
render_opt.line_width = 10.0     # <--- make lines thicker (default is 1.0)
render_opt.background_color = np.array([1, 1, 1])  # optional

vis.run()
vis.destroy_window()

Generating Open3D point cloud with 'rgb' colors...


### Interaction 2 - project lasso to get a 3D plane ###

In [ ]:

def project_lasso_to_3d_fill(
    vignette: ProcessedVignette, 
    uv_lasso: List[Tuple[float, float]]
) -> Tuple[Optional[np.ndarray], Optional[o3d.geometry.TriangleMesh]]:
    """
    Projects a 2D UV lasso to create a 3D boundary curve and a low-poly filled mesh.

    Args:
        vignette: The ProcessedVignette object.
        uv_lasso: A list of (u, v) coordinates forming a closed polygon.

    Returns:
        A tuple containing:
        - The 3D boundary curve (N, 3 NumPy array).
        - The filled 3D mesh (Open3D TriangleMesh).
    """
    print("\n--- Running Test 2: Projecting Lasso to 3D Fill ---")
    
    # 1. Reuse the curve projection logic to get the 3D boundary
    boundary_3d = project_curves_to_3d(vignette, uv_lasso)
    if boundary_3d is None:
        return None, None
        
    # 2. Find all points from the vignette that lie inside the 2D lasso
    try:
        params = _get_projection_params(vignette)
    except ValueError as e:
        print(f"Error: {e}")
        return None, None
        
    vignette_uvs = _project_3d_to_uv(vignette.points, params)
    
    # Use Matplotlib's Path object for efficient point-in-polygon test
    lasso_path = MplPath(uv_lasso)
    inside_mask = lasso_path.contains_points(vignette_uvs)
    
    points_inside_3d = vignette.points[inside_mask]
    
    if len(points_inside_3d) < 3:
        print("Warning: Fewer than 3 points found inside the lasso. Cannot create a mesh.")
        return boundary_3d, None
        
    print(f"Found {len(points_inside_3d)} points inside the lasso.")

    # 3. Create a low-poly fill from the interior points
    # For a "quick test," computing the convex hull is the simplest and most
    # robust way to create a filled mesh that represents the area.
    pcd_inside = o3d.geometry.PointCloud()
    pcd_inside.points = o3d.utility.Vector3dVector(points_inside_3d)
    
    try:
        # The convex hull creates a "shrink-wrapped" mesh around the points.
        filled_mesh, _ = pcd_inside.compute_convex_hull()
        filled_mesh.paint_uniform_color([0.1, 0.8, 0.1]) # Color the mesh green
        print("Successfully created a filled mesh using a convex hull.")
    except Exception as e:
        print(f"Could not compute mesh: {e}")
        return boundary_3d, None
        
    return boundary_3d, filled_mesh
